# Phase 3 — encoder sweep on Modal

Modal equivalent of `kaggle_phase3_sweep.ipynb`. The actual training/probing logic lives in
`modal_app/phase3_sweep.py` as a Modal App — GPU functions have to be defined at module level
to be deployed, so they can't live directly in notebook cells the way Kaggle's could. This
notebook drives that app cell-by-cell, the same way the Kaggle one drove local subprocesses.

## Setup — do this first (once)
1. `pip install modal` and `modal setup` (opens a browser to authenticate this machine).
2. `modal volume create phase3-data` and `modal volume create phase3-results`.
3. **Push your local commits.** The Modal image is built by `git clone`-ing the public repo,
   so uncommitted local changes (e.g. `src/probes/run_sweep.py`, new files) are invisible
   inside the container until pushed.
4. If you already trained some encoders locally or on Kaggle, upload those `backbone.pt` /
   `last_ckpt.pt` files into the `phase3-results` volume — section 2 below.
5. GPU type and how many run in parallel (the Modal analog of Kaggle's "dual T4") are set by
   the `gpu=` / `max_containers=` arguments on `train_cell` in `modal_app/phase3_sweep.py`
   — edit those two lines to change them, then re-run cells below.

## Cost note
Modal bills per-second of actual container run time — there's no idle-quota clock like
Kaggle's 30 h/week. `max_containers=2` caps you at 2 concurrent GPUs so a stray loop can't
silently fan out to 36 containers and blow through budget.

## 1. Verify the GPU

In [ ]:
!modal run modal_app/phase3_sweep.py::verify_gpu

## 2. Upload progress you already have

Restores local `backbone.pt` (completed seed) / `last_ckpt.pt` (partial seed, resumable) into
the `phase3-results` volume, at the same `results/encoders/<run_id>/...` layout the sweep
expects. Run this before `run_sweep` so already-trained seeds are skipped rather than
retrained.

The volume's root is mounted at `results/` inside the container, so the volume-relative path
drops the leading `results/`.

In [ ]:
# One-shot directory upload (recent Modal CLI versions upload directories
# recursively when the local argument is a directory). Check `modal volume put --help`
# first if this errors, and fall back to the per-file loop in the next cell.
!modal volume put phase3-results results/encoders encoders

In [ ]:
# Fallback: per-run-id loop, uploads whichever of backbone.pt / last_ckpt.pt exists.
import subprocess
from pathlib import Path

for run_dir in sorted(Path("results/encoders").glob("*_seed*")):
    rid = run_dir.name
    for fname in ("backbone.pt", "last_ckpt.pt"):
        f = run_dir / fname
        if f.exists():
            remote = f"encoders/{rid}/{fname}"
            print(f"uploading {f} -> phase3-results:{remote}")
            subprocess.run(["modal", "volume", "put", "phase3-results", str(f), remote], check=True)

## 3. Download data + build image caches

Same as the Kaggle notebook's section 4, but writes into the `phase3-data` volume so the
download only has to happen once, not on every new container.

In [ ]:
!modal run modal_app/phase3_sweep.py::download_data

## 4. Run the sweep

Dispatches every not-yet-done `(config, condition, strength, seed)` cell to `train_cell`.
`max_containers=2` on that function caps concurrency at 2 GPUs, mirroring the Kaggle dual-T4
setup — Modal autoscales up to that cap and reuses freed slots as jobs finish, so this call
blocks until the whole grid (or everything up to your budget) is done. Idempotent: each
worker checks the volume for an existing `backbone.pt` and skips if already trained, so
re-running this cell after an interruption just continues.

In [ ]:
!modal run modal_app/phase3_sweep.py::run_sweep

## 5. Progress summary

Re-run any time, including while section 4 is running in another terminal.

In [ ]:
!modal run modal_app/phase3_sweep.py::progress

## 6. Verify the 50-epoch cut on the first encoders

Same sanity check as the Kaggle notebook's section 9, before trusting the whole grid.

In [ ]:
!modal run modal_app/phase3_sweep.py::quality_gate_check

## 7. Probe the trained encoders → stacks.npz

Runs all three cells back-to-back on a single GPU (probing is feature-extraction + small-MLP
fits — no benefit from more than one worker). Skips a cell if its `stacks.npz` already exists,
or if fewer than 12 seeds are trained for it yet.

In [ ]:
!modal run modal_app/phase3_sweep.py::run_probes

## 8. Pull results back down locally

`stacks.npz` / `meta.json` are what the H1–H4 statistics session consumes — get them onto
your machine (or wherever that session runs).

In [ ]:
!modal volume get phase3-results probes results/probes